# USDOT Intersection Safety Challenge — Stage-1B
## Training vs Validation — Structural Comparison

**Purpose:** Determine whether the professor-shared **training** zip files share the same schema,
file types, run structure, and naming conventions as the already-explored **validation** dataset.  
**Constraint:** No full extraction of training archives (~1 TB). Everything is done via `zipfile` inspection
and tiny in-memory sample reads.  
**Output:** A side-by-side comparison table and a research-readiness verdict.

---
## 0 · Setup & Configuration

- **Validation dataset** — downloaded fresh via `wget` into Colab's `/content/` disk each session (skips `.pcap`/`.mp4`). Re-run is a no-op if already extracted.
- **Training dataset** — 4 zip files accessed directly from the professor's shared Google Drive folder. No extraction, no Drive quota consumed.

**Only one line to change:** `SHARED_DRIVE_DIR` in the cell below.

In [1]:
import os, re, io, json, zipfile, warnings, textwrap
from pathlib import Path
from collections import defaultdict

import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.max_colwidth', 55)

print('Imports OK.')

Imports OK.


In [8]:
import zipfile as _zf

# ── 0a · Mount Google Drive & configure paths ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── USER CONFIG — update this one path ───────────────────────────────────
# Shared Drive folder containing the 4 training zips.
# Add a shortcut to My Drive first:
#   Open the folder in Drive → three-dot menu → "Add shortcut to My Drive"
SHARED_DRIVE_DIR = Path('/content/drive/MyDrive/ITS_Intersection_USDOT')

# ── Derived paths (do not edit) ───────────────────────────────────────────
VAL_DOWNLOAD_URL = (
    'https://data.transportation.gov/download/j58q-ymi4/application%2Fx-zip-compressed'
)
VAL_ZIP_LOCAL   = Path('/content/isc_stage1b.zip')
VAL_EXTRACT_DIR = Path('/content/isc_stage1b')
TRAIN_ZIP_DIR   = SHARED_DRIVE_DIR

MAX_SAMPLE_RUNS  = 3
MAX_SAMPLE_FILES = 6

# ── Validate shared Drive folder is reachable ─────────────────────────────
assert SHARED_DRIVE_DIR.exists(), (
    f'Shared Drive folder not found: {SHARED_DRIVE_DIR}\n'
    '  Add a shortcut to My Drive first, or check: !ls /content/drive/Shareddrives/'
)
print(f'Training zip dir  : {TRAIN_ZIP_DIR}')
print(f'Val extract dir   : {VAL_EXTRACT_DIR}  (exists: {VAL_EXTRACT_DIR.exists()})')

# ── Validation — download + extract (skips if already done) ──────────────
# Check if the individual run folders (e.g., Run_410) already exist in /content
# which indicates a previous successful extraction to the root.
if any(Path('/content').glob('Run_*')):
    print('Validation already extracted to /content — skipping download.')
else:
    if not VAL_ZIP_LOCAL.exists():
        print('Downloading validation zip (~28.6 GB) ...')
        !wget -q --show-progress -O /content/isc_stage1b.zip {VAL_DOWNLOAD_URL}
        if not VAL_ZIP_LOCAL.exists():
            raise RuntimeError('wget failed — file not found after download.')
    print(f'Extracting (skipping .pcap / .mp4) ...')
    SKIP_EXT = {'.pcap', '.mp4'}
    with _zf.ZipFile(VAL_ZIP_LOCAL, 'r') as _z:
        _members = [m for m in _z.infolist()
                    if Path(m.filename).suffix.lower() not in SKIP_EXT]
        for _i, _m in enumerate(_members, 1):
            _z.extract(_m, '/content')
            if _i % 500 == 0 or _i == len(_members):
                print(f'  {_i}/{len(_members)} files extracted ...')
    print(f'Done.')

VAL_ROOT = Path('/content') # Corrected: files are extracted directly into /content
print(f'VAL_ROOT      : {VAL_ROOT}')
print(f'TRAIN_ZIP_DIR : {TRAIN_ZIP_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training zip dir  : /content/drive/MyDrive/ITS_Intersection_USDOT
Val extract dir   : /content/isc_stage1b  (exists: False)
Validation already extracted to /content — skipping download.
VAL_ROOT      : /content
TRAIN_ZIP_DIR : /content/drive/MyDrive/ITS_Intersection_USDOT


In [4]:
print('Setup complete.')

Setup complete.


In [5]:
# ── Shared helpers ────────────────────────────────────────────────────────

def fmt_size(n_bytes: int) -> str:
    """Human-readable file size."""
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if n_bytes < 1024:
            return f'{n_bytes:.1f} {unit}'
        n_bytes /= 1024
    return f'{n_bytes:.1f} PB'


def read_bytes_from_zip(zf: zipfile.ZipFile, member_name: str, n_bytes: int = 8192) -> bytes:
    """Read the first n_bytes from an archive member without extracting it."""
    with zf.open(member_name) as f:
        return f.read(n_bytes)


def csv_head_from_bytes(raw: bytes, nrows: int = 5) -> pd.DataFrame:
    """Parse first nrows of CSV from a bytes buffer."""
    return pd.read_csv(io.BytesIO(raw), nrows=nrows, on_bad_lines='skip')


def csv_head_no_header(raw: bytes, nrows: int = 5) -> pd.DataFrame:
    """Parse headerless CSV from bytes."""
    return pd.read_csv(io.BytesIO(raw), header=None, nrows=nrows, on_bad_lines='skip')


def json_sample_from_bytes(raw: bytes):
    """Parse JSON or return first 600 chars as string if parsing fails."""
    try:
        return json.loads(raw)
    except Exception:
        return raw[:600].decode('utf-8', errors='replace')


RUN_RE = re.compile(r'Run_\d+', re.IGNORECASE)

print('Helpers defined.')

Helpers defined.


---
## 1 · Validation Dataset — Baseline Summary

Quickly enumerate what we already know about the validation set so we have a baseline for comparison.

In [9]:
# ── Enumerate validation runs ─────────────────────────────────────────────
val_runs = sorted([d for d in VAL_ROOT.iterdir() if d.is_dir() and RUN_RE.match(d.name)])
print(f'Validation runs found : {len(val_runs)}')
print('Sample run names      :', [r.name for r in val_runs[:6]], '...')

# ── Per-run file type inventory (confirmed from notebook 01) ──────────────
VAL_KNOWN = {
    'GT csv'              : re.compile(r'Run_\d+_GT\.csv$'),
    'ISC_all_timing csv'  : re.compile(r'ISC_Run_\d+_ISC_all_timing\.csv$'),
    'v2xhub_timing csv'   : re.compile(r'ISC_Run_\d+_v2xhub_timing\.csv$'),
    'VisualCamera timing' : re.compile(r'VisualCamera\d+_Run_\d+_frame-timing\.csv$'),
    'ThermalCamera timing': re.compile(r'ThermalCamera\d+_Run_\d+_frame-timing\.csv$'),
    'V2XHubSensor csv'    : re.compile(r'V2XHubSensor_Run_\d+\.csv$'),
    'Radar sensor json'   : re.compile(r'Radars_Run_\d+_sensor\d+\.json$'),
    'Traffic triggers json': re.compile(r'Radars_Run_\d+_traffic-triggers-output\.json$'),
}

val_type_counts = defaultdict(int)
for run in val_runs:
    for f in run.iterdir():
        for label, pat in VAL_KNOWN.items():
            if pat.search(f.name):
                val_type_counts[label] += 1

# Root-level labels file
labels_file = VAL_ROOT / 'conflict_no_conflict_labels_all_validation_runs.csv'
val_type_counts['Labels csv'] = 1 if labels_file.exists() else 0

print()
print('─'*55)
print(f'{"File type":<28} {"Count (across all runs)":>20}')
print('─'*55)
for label, cnt in val_type_counts.items():
    print(f'  {label:<26} {cnt:>20}')
print('─'*55)

Validation runs found : 41
Sample run names      : ['Run_1003', 'Run_1023', 'Run_1027', 'Run_1078', 'Run_108', 'Run_1169'] ...

───────────────────────────────────────────────────────
File type                    Count (across all runs)
───────────────────────────────────────────────────────
  ThermalCamera timing                        205
  VisualCamera timing                         327
  GT csv                                       41
  ISC_all_timing csv                           41
  v2xhub_timing csv                            30
  Radar sensor json                            84
  Traffic triggers json                        21
  V2XHubSensor csv                             30
  Labels csv                                    1
───────────────────────────────────────────────────────


---
## 2 · Training Zip Files — Discovery & Inventory

List all zip files, print their sizes, and inspect archive contents without extracting anything.

In [10]:
# ── Find all zip files ────────────────────────────────────────────────────
zip_paths = sorted(TRAIN_ZIP_DIR.glob('*.zip'))

if not zip_paths:
    print('⚠  No .zip files found in TRAIN_ZIP_DIR. Check your path.')
else:
    print(f'Found {len(zip_paths)} zip file(s):\n')
    total_bytes = 0
    for zp in zip_paths:
        sz = zp.stat().st_size
        total_bytes += sz
        print(f'  {zp.name:<35}  {fmt_size(sz):>10}')
    print(f'\n  Total size : {fmt_size(total_bytes)}')

Found 4 zip file(s):

  Training Data 1.zip                    140.7 GB
  Training Data 2.zip                    140.2 GB
  Training Data 3.zip                    142.1 GB
  Training Data 4.zip                    139.4 GB

  Total size : 562.4 GB


In [11]:
# ── Inspect each archive (no extraction) ──────────────────────────────────
# For each zip: list Run_* folders, count them, sample file paths.

train_zip_meta = {}   # keyed by zip name

for zp in zip_paths:
    print('=' * 65)
    print(f'  ZIP: {zp.name}  ({fmt_size(zp.stat().st_size)})')
    print('=' * 65)

    try:
        with zipfile.ZipFile(zp, 'r') as zf:
            all_names = zf.namelist()          # all member paths
            info_list = zf.infolist()

        total_members = len(all_names)

        # Top-level names (no subdirectory)
        top_level = sorted(set(
            n.split('/')[0] for n in all_names if n.strip()
        ))

        # Run_* folders
        run_folders = sorted(set(
            n.split('/')[0] for n in all_names
            if RUN_RE.match(n.split('/')[0])
        ))

        # Extension frequency
        ext_counts = defaultdict(int)
        for n in all_names:
            ext = Path(n).suffix.lower()
            if ext:
                ext_counts[ext] += 1

        print(f'  Total archive members : {total_members:,}')
        print(f'  Top-level entries     : {len(top_level)}')
        print(f'  Run_* folders found   : {len(run_folders)}')
        print(f'  Sample run names      : {run_folders[:MAX_SAMPLE_RUNS]} ...')
        print()
        print('  Extension breakdown:')
        for ext, cnt in sorted(ext_counts.items(), key=lambda x: -x[1]):
            print(f'    {ext:<10}  {cnt:>6}')
        print()
        print(f'  Sample internal paths (first {MAX_SAMPLE_FILES}):')
        for n in all_names[:MAX_SAMPLE_FILES]:
            print(f'    {n}')
        print()

        train_zip_meta[zp.name] = {
            'path'         : zp,
            'total_members': total_members,
            'run_folders'  : run_folders,
            'ext_counts'   : dict(ext_counts),
            'all_names'    : all_names,
        }

    except zipfile.BadZipFile as e:
        print(f'  ✗ Could not open archive: {e}')
        train_zip_meta[zp.name] = {'error': str(e)}

print('Done inspecting zip headers.')

  ZIP: Training Data 1.zip  (140.7 GB)
  Total archive members : 6,948
  Top-level entries     : 200
  Run_* folders found   : 199
  Sample run names      : ['Run_1001', 'Run_1006', 'Run_1008'] ...

  Extension breakdown:
    .csv          3083
    .mp4          2600
    .pcap          540
    .json          525

  Sample internal paths (first 6):
    Run_166/
    Run_166/VisualCamera3_Run_166.mp4
    Run_166/Radars_Run_166_traffic-triggers-output.json
    Run_166/Radars_Run_166_sensor1.json
    Run_166/VisualCamera1_Run_166.mp4
    Run_166/ThermalCamera5_Run_166.mp4

  ZIP: Training Data 2.zip  (140.2 GB)
  Total archive members : 6,930
  Top-level entries     : 200
  Run_* folders found   : 200
  Sample run names      : ['Run_1000', 'Run_1005', 'Run_1010'] ...

  Extension breakdown:
    .csv          3090
    .mp4          2600
    .pcap          545
    .json          495

  Sample internal paths (first 6):
    Run_1306/
    Run_1306/VisualCamera1_Run_1306.mp4
    Run_1306/VisualCa

---
## 3 · Folder Structure Comparison

Compare naming conventions, run-folder style, and file-type presence between training and validation.

In [12]:
# ── Aggregate all training archive members ────────────────────────────────
all_train_names = []
train_run_folders_all = set()
train_ext_counts_all = defaultdict(int)

for zname, meta in train_zip_meta.items():
    if 'error' in meta:
        continue
    all_train_names.extend(meta['all_names'])
    train_run_folders_all.update(meta['run_folders'])
    for ext, cnt in meta['ext_counts'].items():
        train_ext_counts_all[ext] += cnt

train_run_folders_sorted = sorted(train_run_folders_all)

print(f'Total training archive members (all zips) : {len(all_train_names):,}')
print(f'Total unique Run_* folders across all zips : {len(train_run_folders_sorted)}')
print(f'Sample run names : {train_run_folders_sorted[:8]} ...')
print()
print('Training extension totals:')
for ext, cnt in sorted(train_ext_counts_all.items(), key=lambda x: -x[1]):
    print(f'  {ext:<10}  {cnt:>7}')

Total training archive members (all zips) : 27,664
Total unique Run_* folders across all zips : 799
Sample run names : ['Run_1', 'Run_10', 'Run_100', 'Run_1000', 'Run_1001', 'Run_1002', 'Run_1004', 'Run_1005'] ...

Training extension totals:
  .csv          12337
  .mp4          10400
  .pcap          2167
  .json          1960


In [13]:
# ── Check presence of each known file type inside training archives ────────
TRAIN_PATTERNS = {
    'GT csv'               : re.compile(r'Run_\d+_GT\.csv$'),
    'ISC_all_timing csv'   : re.compile(r'ISC_Run_\d+_ISC_all_timing\.csv$'),
    'v2xhub_timing csv'    : re.compile(r'ISC_Run_\d+_v2xhub_timing\.csv$'),
    'VisualCamera timing'  : re.compile(r'VisualCamera\d+_Run_\d+_frame-timing\.csv$'),
    'ThermalCamera timing' : re.compile(r'ThermalCamera\d+_Run_\d+_frame-timing\.csv$'),
    'V2XHubSensor csv'     : re.compile(r'V2XHubSensor_Run_\d+\.csv$'),
    'Radar sensor json'    : re.compile(r'Radars_Run_\d+_sensor\d+\.json$'),
    'Traffic triggers json': re.compile(r'Radars_Run_\d+_traffic-triggers-output\.json$'),
    'Labels csv'           : re.compile(r'labels.*\.csv$', re.IGNORECASE),
}

train_type_counts = defaultdict(int)
for n in all_train_names:
    fname = Path(n).name
    for label, pat in TRAIN_PATTERNS.items():
        if pat.search(fname):
            train_type_counts[label] += 1

print('File type presence in training archives:')
print(f'{"File type":<28} {"Val count":>10} {"Train count":>12} {"Present in train?":>18}')
print('─' * 72)
for label in TRAIN_PATTERNS:
    v_cnt = val_type_counts.get(label, 0)
    t_cnt = train_type_counts.get(label, 0)
    present = '✓ Yes' if t_cnt > 0 else '✗ Missing'
    print(f'  {label:<26} {v_cnt:>10} {t_cnt:>12} {present:>18}')
print('─' * 72)

File type presence in training archives:
File type                     Val count  Train count  Present in train?
────────────────────────────────────────────────────────────────────────
  GT csv                             41            4              ✓ Yes
  ISC_all_timing csv                 41          799              ✓ Yes
  v2xhub_timing csv                  30          566              ✓ Yes
  VisualCamera timing               327         6391              ✓ Yes
  ThermalCamera timing              205         3995              ✓ Yes
  V2XHubSensor csv                   30          566              ✓ Yes
  Radar sensor json                  84         1564              ✓ Yes
  Traffic triggers json              21          391              ✓ Yes
  Labels csv                          1            0          ✗ Missing
────────────────────────────────────────────────────────────────────────


In [14]:
# ── Run name style comparison ─────────────────────────────────────────────
val_run_ids  = sorted(int(RUN_RE.search(r.name).group().split('_')[1])
                      for r in val_runs if RUN_RE.search(r.name))
train_run_ids = sorted(int(RUN_RE.search(r).group().split('_')[1])
                       for r in train_run_folders_sorted
                       if RUN_RE.search(r))

overlap = set(val_run_ids) & set(train_run_ids)

print('Run-folder naming style')
print(f'  Validation  — count: {len(val_run_ids):>4}  range: {min(val_run_ids)}–{max(val_run_ids)}')
print(f'  Training    — count: {len(train_run_ids):>4}  range: {min(train_run_ids) if train_run_ids else "??"}–{max(train_run_ids) if train_run_ids else "??"}')
print(f'  Overlapping run IDs: {len(overlap)}  (should be 0 for non-overlapping splits)')
if overlap:
    print(f'  ⚠  Shared IDs: {sorted(overlap)[:10]}')
else:
    print('  ✓  No overlap — training and validation runs are distinct.')
print()
print(f'  Pattern match Run_XXXX format?')
all_match_val   = all(RUN_RE.fullmatch(r.name) for r in val_runs)
all_match_train = all(RUN_RE.fullmatch(r) for r in train_run_folders_sorted)
print(f'    Validation : {"✓ all match" if all_match_val   else "✗ some differ"}')
print(f'    Training   : {"✓ all match" if all_match_train else "✗ some differ"}')

Run-folder naming style
  Validation  — count:   41  range: 48–1290
  Training    — count:  799  range: 1–1319
  Overlapping run IDs: 11  (should be 0 for non-overlapping splits)
  ⚠  Shared IDs: [48, 55, 91, 737, 761, 849, 850, 1023, 1078, 1228]

  Pattern match Run_XXXX format?
    Validation : ✓ all match
    Training   : ✓ all match


---
## 4 · Lightweight Schema Comparison

For each key file type, read a tiny sample (~8 KB) directly from the zip and compare columns,
JSON keys, and timestamp formats against the known validation schema.

In [15]:
# ── Pick one accessible zip for sampling ─────────────────────────────────
# Use the first non-errored zip
sample_zip_name = next(
    (zn for zn, m in train_zip_meta.items() if 'error' not in m), None
)

if sample_zip_name is None:
    raise RuntimeError('No valid zip available for sampling.')

sample_zip_path   = train_zip_meta[sample_zip_name]['path']
sample_all_names  = train_zip_meta[sample_zip_name]['all_names']

def find_member(names, pattern: re.Pattern):
    """Return the first archive member matching pattern."""
    for n in names:
        if pattern.search(Path(n).name):
            return n
    return None

print(f'Using zip for samples : {sample_zip_name}')

MEMBER_GT         = find_member(sample_all_names, re.compile(r'Run_\d+_GT\.csv$'))
MEMBER_V2X        = find_member(sample_all_names, re.compile(r'V2XHubSensor_Run_\d+\.csv$'))
MEMBER_ISC        = find_member(sample_all_names, re.compile(r'ISC_Run_\d+_ISC_all_timing\.csv$'))
MEMBER_RADAR      = find_member(sample_all_names, re.compile(r'Radars_Run_\d+_sensor\d+\.json$'))
MEMBER_TRIGGERS   = find_member(sample_all_names, re.compile(r'Radars_Run_\d+_traffic-triggers-output\.json$'))
MEMBER_LABELS     = find_member(sample_all_names, re.compile(r'labels.*\.csv$', re.IGNORECASE))

for label, m in [
    ('GT csv'         , MEMBER_GT),
    ('V2X csv'        , MEMBER_V2X),
    ('ISC timing csv' , MEMBER_ISC),
    ('Radar json'     , MEMBER_RADAR),
    ('Triggers json'  , MEMBER_TRIGGERS),
    ('Labels csv'     , MEMBER_LABELS),
]:
    status = f'✓  {m}' if m else '✗  NOT FOUND'
    print(f'  {label:<20}  {status}')

Using zip for samples : Training Data 1.zip
  GT csv                ✓  Run_1078/Run_1078_GT.csv
  V2X csv               ✓  Run_166/V2XHubSensor_Run_166.csv
  ISC timing csv        ✓  Run_166/ISC_Run_166_ISC_all_timing.csv
  Radar json            ✓  Run_166/Radars_Run_166_sensor1.json
  Triggers json         ✓  Run_166/Radars_Run_166_traffic-triggers-output.json
  Labels csv            ✗  NOT FOUND


In [16]:
# ── 4a. GT CSV ─────────────────────────────────────────────────────────────
print('── GT CSV schema ──────────────────────────────────────────')

if MEMBER_GT:
    with zipfile.ZipFile(sample_zip_path, 'r') as zf:
        raw_gt = read_bytes_from_zip(zf, MEMBER_GT, 32_768)

    df_gt = csv_head_from_bytes(raw_gt, nrows=3)
    cols_gt = list(df_gt.columns)

    # Expected validation pattern: first col = 'Time', rest = CLASS_field
    has_time_col = cols_gt[0].lower() == 'time' if cols_gt else False
    wide_pattern = all(re.search(r'_', c) for c in cols_gt[1:] if c) if len(cols_gt) > 1 else False

    print(f'  Member       : {MEMBER_GT}')
    print(f'  Shape sample : {df_gt.shape}')
    print(f'  First col    : {cols_gt[0] if cols_gt else "??"}')
    print(f'  Has Time col : {has_time_col}')
    print(f'  Wide format  : {wide_pattern}')
    print(f'  Col sample   : {cols_gt[:8]}')
    print(f'  Time values  : {df_gt.iloc[:,0].tolist()}')
    print()
    print('  Training GT head:')
    display(df_gt.head(3))
else:
    print('  ✗ No GT csv member found in sample zip.')

── GT CSV schema ──────────────────────────────────────────
  Member       : Run_1078/Run_1078_GT.csv
  Shape sample : (3, 37)
  First col    : Time
  Has Time col : True
  Wide format  : True
  Col sample   : ['Time', 'Passenger_Vehicle_xctr', 'Passenger_Vehicle_yctr', 'Passenger_Vehicle_zctr', 'Passenger_Vehicle_xlen', 'Passenger_Vehicle_ylen', 'Passenger_Vehicle_zlen', 'Passenger_Vehicle_xrot']
  Time values  : [1701988419.17, 1701988419.27, 1701988419.37]

  Training GT head:


,Time,Passenger_Vehicle_xctr,Passenger_Vehicle_yctr,Passenger_Vehicle_zctr,Passenger_Vehicle_xlen,Passenger_Vehicle_ylen,Passenger_Vehicle_zlen,Passenger_Vehicle_xrot,Passenger_Vehicle_yrot,Passenger_Vehicle_zrot,Passenger_Vehicle_xctr.1,Passenger_Vehicle_yctr.1,Passenger_Vehicle_zctr.1,Passenger_Vehicle_xlen.1,Passenger_Vehicle_ylen.1,Passenger_Vehicle_zlen.1,Passenger_Vehicle_xrot.1,Passenger_Vehicle_yrot.1,Passenger_Vehicle_zrot.1,VRU_Adult_Using_Motorized Bicycle_xctr,VRU_Adult_Using_Motorized Bicycle_yctr,VRU_Adult_Using_Motorized Bicycle_zctr,VRU_Adult_Using_Motorized Bicycle_xlen,VRU_Adult_Using_Motorized Bicycle_ylen,VRU_Adult_Using_Motorized Bicycle_zlen,VRU_Adult_Using_Motorized Bicycle_xrot,VRU_Adult_Using_Motorized Bicycle_yrot,VRU_Adult_Using_Motorized Bicycle_zrot,VRU_Adult_xctr,VRU_Adult_yctr,VRU_Adult_zctr,VRU_Adult_xlen,VRU_Adult_ylen,VRU_Adult_zlen,VRU_Adult_xrot,VRU_Adult_yrot,VRU_Adult_zrot
0,1.701988e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.912764,-25.419045,-3.342802,1.255006,1.771847,2.010049,0,0,341.454206,20.652902,4.084841,-3.543396,1.12889,0.94439,1.769067,0,0,181.316902
1,1.701988e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.912764,-25.419045,-3.342802,1.255006,1.771847,2.010049,0,0,341.454206,20.652902,4.084841,-3.543396,1.12889,0.94439,1.769067,0,0,181.316902
2,1.701988e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.912764,-25.419045,-3.342802,1.255006,1.771847,2.010049,0,0,341.454206,20.652902,4.084841,-3.543396,1.12889,0.94439,1.769067,0,0,181.316902


In [17]:
# ── 4b. V2XHubSensor CSV ───────────────────────────────────────────────────
print('── V2XHubSensor CSV schema ────────────────────────────────')

if MEMBER_V2X:
    with zipfile.ZipFile(sample_zip_path, 'r') as zf:
        raw_v2x = read_bytes_from_zip(zf, MEMBER_V2X, 8_192)

    # Try with header first
    df_v2x_hdr = csv_head_from_bytes(raw_v2x, nrows=4)
    # Try without header (expected correct approach for validation)
    df_v2x_raw = csv_head_no_header(raw_v2x, nrows=4)

    n_cols_hdr = df_v2x_hdr.shape[1]
    n_cols_raw = df_v2x_raw.shape[1]

    print(f'  Member          : {MEMBER_V2X}')
    print(f'  Cols WITH header parse  : {n_cols_hdr}  (expected 3, bug if 4)')
    print(f'  Cols WITHOUT header     : {n_cols_raw}  (expected 4 — unquoted [2,8] splits)')
    print()
    print('  Raw parse (no header):')
    display(df_v2x_raw)

    # Apply the known fix
    if n_cols_raw >= 3:
        n = n_cols_raw
        df_v2x_fix = pd.DataFrame({
            'timestamp_ms'      : df_v2x_raw.iloc[:, 0],
            'active_signal_ids' : df_v2x_raw.iloc[:, 1:n-1].astype(str).apply(','.join, axis=1).str.strip(),
            'event_list'        : df_v2x_raw.iloc[:, n-1],
        })
        print('  After merge-fix:')
        display(df_v2x_fix)
else:
    print('  ✗ No V2XHubSensor csv found in sample zip.')

── V2XHubSensor CSV schema ────────────────────────────────
  Member          : Run_166/V2XHubSensor_Run_166.csv
  Cols WITH header parse  : 4  (expected 3, bug if 4)
  Cols WITHOUT header     : 4  (expected 4 — unquoted [2,8] splits)

  Raw parse (no header):


,0,1,2,3
0,1708095218267,[2,8],[]
1,1708095219267,[2,8],[]
2,1708095220266,[2,8],[]
3,1708095221266,[2,8],[]


  After merge-fix:


,timestamp_ms,active_signal_ids,event_list
0,1708095218267,"[2, 8]",[]
1,1708095219267,"[2, 8]",[]
2,1708095220266,"[2, 8]",[]
3,1708095221266,"[2, 8]",[]


In [18]:
# ── 4c. ISC All-timing CSV ─────────────────────────────────────────────────
print('── ISC All-timing CSV schema ──────────────────────────────')

if MEMBER_ISC:
    with zipfile.ZipFile(sample_zip_path, 'r') as zf:
        raw_isc = read_bytes_from_zip(zf, MEMBER_ISC, 4_096)

    df_isc_hdr = csv_head_from_bytes(raw_isc, nrows=5)
    df_isc_raw = csv_head_no_header(raw_isc, nrows=5)

    EXPECTED_ISC_COLS = ['sensor_type', 'sensor_name', 'ip_address',
                         'filename', 'start_time', 'end_time', 'duration']

    print(f'  Member             : {MEMBER_ISC}')
    print(f'  Cols w/ header     : {list(df_isc_hdr.columns)}')
    print(f'  Cols w/o header    : {list(df_isc_raw.columns)}')
    print(f'  Expected cols      : {EXPECTED_ISC_COLS}')
    print(f'  Is headerless?     : {list(df_isc_hdr.columns) != EXPECTED_ISC_COLS}')
    print()
    print('  Raw parse (no header):')
    display(df_isc_raw)

    # Check start_time / end_time format
    if df_isc_raw.shape[1] >= 7:
        sample_ts = str(df_isc_raw.iloc[0, 4])
        print(f'  start_time sample  : {sample_ts}')
        is_datetime_fmt = bool(re.match(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', sample_ts))
        print(f'  Matches YYYY-MM-DD HH:MM:SS format: {is_datetime_fmt}')
else:
    print('  ✗ No ISC timing csv found in sample zip.')

── ISC All-timing CSV schema ──────────────────────────────
  Member             : Run_166/ISC_Run_166_ISC_all_timing.csv
  Cols w/ header     : ['radar_camera', 'VisualCamera7', '192.168.55.44:53554', 'VisualCamera7_S02C01R1_1.mp4', '2024-02-16 09:53:41.283802', '2024-02-16 09:54:34.974784', '0:00:53.682672']
  Cols w/o header    : [0, 1, 2, 3, 4, 5, 6]
  Expected cols      : ['sensor_type', 'sensor_name', 'ip_address', 'filename', 'start_time', 'end_time', 'duration']
  Is headerless?     : True

  Raw parse (no header):


,0,1,2,3,4,5,6
0,radar_camera,VisualCamera7,192.168.55.44:53554,VisualCamera7_S02C01R1_1.mp4,2024-02-16 09:53:41.283802,2024-02-16 09:54:34.974784,0:00:53.682672
1,flir,ThermalCamera2,192.168.55.181,ThermalCamera2_S02C01R1_1.mp4,2024-02-16 09:53:39.759557,2024-02-16 09:54:34.979984,0:00:55.176050
2,axis,VisualCamera4,192.168.55.30,VisualCamera4_S02C01R1_1.mp4,2024-02-16 09:53:40.208124,2024-02-16 09:54:34.977146,0:00:54.750852
3,radar_camera,VisualCamera5,192.168.55.44:51554,VisualCamera5_S02C01R1_1.mp4,2024-02-16 09:53:40.823207,2024-02-16 09:54:34.983620,0:00:54.130853
4,flir,ThermalCamera1,192.168.55.180,ThermalCamera1_S02C01R1_1.mp4,2024-02-16 09:53:39.771219,2024-02-16 09:54:34.990703,0:00:55.168776


  start_time sample  : 2024-02-16 09:53:41.283802
  Matches YYYY-MM-DD HH:MM:SS format: True


In [19]:
# ── 4d. Radar Sensor JSON ──────────────────────────────────────────────────
print('── Radar Sensor JSON schema ───────────────────────────────')

if MEMBER_RADAR:
    with zipfile.ZipFile(sample_zip_path, 'r') as zf:
        raw_radar = read_bytes_from_zip(zf, MEMBER_RADAR, 32_768)

    parsed_radar = json_sample_from_bytes(raw_radar)

    if isinstance(parsed_radar, list) and len(parsed_radar) > 0:
        rec0 = parsed_radar[0]
        top_keys   = list(rec0.keys())
        payload    = rec0.get('payload', {})
        pay_keys   = list(payload.keys()) if isinstance(payload, dict) else []
        topic_val  = rec0.get('topic', '??')
        ts_val     = payload.get('timestamp', '??') if isinstance(payload, dict) else '??'

        EXPECTED_TOP  = ['topic', 'payload', 'qos', 'receivedAt', 'retain']
        EXPECTED_PAY  = ['timestamp', 'cycle_time', 'reference_name', 'objects']

        print(f'  Member          : {MEMBER_RADAR}')
        print(f'  Records in sample chunk : {len(parsed_radar)}')
        print(f'  Top-level keys  : {top_keys}')
        print(f'  Expected        : {EXPECTED_TOP}')
        print(f'  Key match       : {set(top_keys) >= set(EXPECTED_TOP)}')
        print(f'  payload keys    : {pay_keys}')
        print(f'  topic value     : {topic_val}')
        print(f'  timestamp value : {ts_val}')
        is_iso = bool(re.match(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}', str(ts_val)))
        print(f'  ISO 8601 ts?    : {is_iso}')

        objects = payload.get('objects', []) if isinstance(payload, dict) else []
        if objects:
            obj0 = objects[0]
            print(f'  Object keys     : {list(obj0.keys())}')
    else:
        print('  Could not parse as list — raw preview:')
        print(str(parsed_radar)[:400])
else:
    print('  ✗ No radar sensor json found in sample zip.')

── Radar Sensor JSON schema ───────────────────────────────
  Could not parse as list — raw preview:
[
    {
        "topic": "sensor-traffic-objects/sensor1",
        "payload": {
            "cycle_time": 0.10000299662351608,
            "objects": [],
            "reference_name": "sensor1",
            "timestamp": "2024-02-16T14:55:51+00:00"
        },
        "qos": "AT_LEAST_ONCE",
        "receivedAt": "2024-02-16 09:53:40",
        "retain": true
    },
    {
        "topic": "sensor-dia


In [20]:
# ── 4e. Traffic-triggers JSON ──────────────────────────────────────────────
print('── Traffic-Triggers JSON schema ───────────────────────────')

if MEMBER_TRIGGERS:
    with zipfile.ZipFile(sample_zip_path, 'r') as zf:
        raw_trig = read_bytes_from_zip(zf, MEMBER_TRIGGERS, 32_768)

    parsed_trig = json_sample_from_bytes(raw_trig)

    if isinstance(parsed_trig, list) and len(parsed_trig) > 0:
        rec0 = parsed_trig[0]
        top_keys = list(rec0.keys())
        payload  = rec0.get('payload', {})
        pay_keys = list(payload.keys()) if isinstance(payload, dict) else []
        ts_val   = payload.get('timestamp', '??') if isinstance(payload, dict) else '??'

        EXPECTED_TOP = ['topic', 'payload', 'qos', 'receivedAt', 'retain']
        EXPECTED_PAY = ['timestamp', 'trigger_outputs']

        print(f'  Member         : {MEMBER_TRIGGERS}')
        print(f'  Records in chunk : {len(parsed_trig)}')
        print(f'  Top keys       : {top_keys}')
        print(f'  Key match      : {set(top_keys) >= set(EXPECTED_TOP)}')
        print(f'  payload keys   : {pay_keys}')
        print(f'  timestamp      : {ts_val}')

        trig_outs = payload.get('trigger_outputs', []) if isinstance(payload, dict) else []
        if trig_outs:
            to0 = trig_outs[0]
            tts = to0.get('traffic_triggers', [])
            if tts:
                print(f'  trigger keys   : {list(tts[0].keys())}')
    else:
        print('  Could not parse — raw preview:')
        print(str(parsed_trig)[:400])
else:
    print('  ✗ No traffic-triggers json found in sample zip.')

── Traffic-Triggers JSON schema ───────────────────────────
  Could not parse — raw preview:
[
    {
        "topic": "traffic-triggers-output",
        "payload": {
            "timestamp": "2024-02-16T14:55:51+00:00",
            "trigger_outputs": [
                {
                    "traffic_triggers": [
                        {
                            "associated_lane": "lane22",
                            "associated_sensor": "sensor3",
                            "associat


In [21]:
# ── 4f. Labels file ────────────────────────────────────────────────────────
print('── Labels CSV schema ──────────────────────────────────────')

# Validation
if labels_file.exists():
    df_val_labels = pd.read_csv(labels_file)
    print('  Validation labels:')
    print(f'    Columns : {list(df_val_labels.columns)}')
    print(f'    Rows    : {len(df_val_labels)}')
    print(f'    Values  : {df_val_labels.iloc[:, 1].unique().tolist()}')
else:
    print('  ✗ Validation labels file not found at expected path.')

print()

# Training
if MEMBER_LABELS:
    with zipfile.ZipFile(sample_zip_path, 'r') as zf:
        raw_lbl = read_bytes_from_zip(zf, MEMBER_LABELS, 8_192)
    df_train_labels = csv_head_from_bytes(raw_lbl, nrows=10)
    print('  Training labels:')
    print(f'    Member  : {MEMBER_LABELS}')
    print(f'    Columns : {list(df_train_labels.columns)}')
    display(df_train_labels.head(5))
else:
    print('  ✗ No labels csv found in training archives.')
    print('    Note: Training may NOT include conflict labels (ground truth held by organizers).')

── Labels CSV schema ──────────────────────────────────────
  Validation labels:
    Columns : ['Run ID', 'Conflict_No_Conflict_Label']
    Rows    : 41
    Values  : ['no conflict', 'conflict']

  ✗ No labels csv found in training archives.
    Note: Training may NOT include conflict labels (ground truth held by organizers).


---
## 5 · Summary Comparison Table

Side-by-side verdict for every dataset component.

In [22]:
# ── Build the comparison table ─────────────────────────────────────────────
# Results are partially populated from the cells above;
# the flags below will be overridden by actual evidence where available.

def _yn(condition, partial=False):
    if partial:
        return 'Partial'
    return 'Yes' if condition else 'No'


# Gather evidence from earlier cells (safe defaults)
gt_present        = bool(MEMBER_GT)
v2x_present       = bool(MEMBER_V2X)
isc_present       = bool(MEMBER_ISC)
radar_present     = bool(MEMBER_RADAR)
triggers_present  = bool(MEMBER_TRIGGERS)
labels_present    = bool(MEMBER_LABELS)

run_style_match   = all(RUN_RE.fullmatch(r) for r in train_run_folders_sorted) if train_run_folders_sorted else False

rows = [
    {
        'component'        : 'Run folders',
        'validation_format': 'Run_XXXX/ (41 runs)',
        'training_format'  : f'Run_XXXX/ ({len(train_run_folders_sorted)} runs detected)',
        'same'             : _yn(run_style_match),
        'notes'            : 'Naming pattern consistent' if run_style_match else 'Pattern differs — check manually',
        'action_needed'    : 'None' if run_style_match else 'Update RUN_RE pattern',
    },
    {
        'component'        : 'GT csv',
        'validation_format': 'Run_XXXX_GT.csv — Time + CLASS_field cols, headerless check OK',
        'training_format'  : 'Run_XXXX_GT.csv — see schema cell 4a' if gt_present else 'NOT FOUND',
        'same'             : _yn(gt_present),
        'notes'            : 'Wide-format confirmed present' if gt_present else 'Missing — verify zip path',
        'action_needed'    : 'None — reuse GT parser' if gt_present else 'Investigate missing GT',
    },
    {
        'component'        : 'Camera timing (Visual)',
        'validation_format': 'VisualCameraX_Run_XXXX_frame-timing.csv  [Image_number, Timestamp]',
        'training_format'  : 'Present' if train_type_counts.get('VisualCamera timing', 0) > 0 else 'NOT FOUND',
        'same'             : _yn(train_type_counts.get('VisualCamera timing', 0) > 0),
        'notes'            : '8 cameras per run expected',
        'action_needed'    : 'None' if train_type_counts.get('VisualCamera timing', 0) > 0 else 'Check if cameras renamed',
    },
    {
        'component'        : 'Camera timing (Thermal)',
        'validation_format': 'ThermalCameraX_Run_XXXX_frame-timing.csv  [Image_number, Timestamp]',
        'training_format'  : 'Present' if train_type_counts.get('ThermalCamera timing', 0) > 0 else 'NOT FOUND',
        'same'             : _yn(train_type_counts.get('ThermalCamera timing', 0) > 0),
        'notes'            : '5 thermal cameras per run expected',
        'action_needed'    : 'None' if train_type_counts.get('ThermalCamera timing', 0) > 0 else 'Check naming',
    },
    {
        'component'        : 'ISC all-timing csv',
        'validation_format': 'Headerless, 7 cols: sensor_type … duration',
        'training_format'  : 'Present — see schema cell 4c' if isc_present else 'NOT FOUND',
        'same'             : _yn(isc_present),
        'notes'            : 'Headerless parse required',
        'action_needed'    : 'None — reuse header=None parser' if isc_present else 'Investigate',
    },
    {
        'component'        : 'V2XHubSensor csv',
        'validation_format': 'Headerless, 4 raw cols (unquoted [2,8] splits col 2)',
        'training_format'  : 'Present — see schema cell 4b' if v2x_present else 'NOT FOUND',
        'same'             : _yn(v2x_present),
        'notes'            : '4-col merge-fix required; only in runs with V2X hardware',
        'action_needed'    : 'None — reuse merge-fix' if v2x_present else 'Verify if V2X hardware used in training',
    },
    {
        'component'        : 'Radar sensor JSON',
        'validation_format': 'List of records with topic/payload/qos/receivedAt/retain',
        'training_format'  : 'Present — see schema cell 4d' if radar_present else 'NOT FOUND',
        'same'             : _yn(radar_present),
        'notes'            : 'ISO 8601 UTC timestamps; filter to traffic-objects topic',
        'action_needed'    : 'None — reuse radar parser' if radar_present else 'Investigate',
    },
    {
        'component'        : 'Traffic-triggers JSON',
        'validation_format': 'List of records with trigger_outputs → traffic_triggers list',
        'training_format'  : 'Present — see schema cell 4e' if triggers_present else 'NOT FOUND',
        'same'             : _yn(triggers_present),
        'notes'            : '~2500 records per run; flatten to zone/lane/sensor rows',
        'action_needed'    : 'None — reuse triggers parser' if triggers_present else 'Investigate',
    },
    {
        'component'        : 'Labels csv',
        'validation_format': 'conflict_no_conflict_labels_all_validation_runs.csv  [Run ID, Label]',
        'training_format'  : 'Present in archive' if labels_present else 'NOT FOUND (expected — withheld)',
        'same'             : _yn(labels_present, partial=True) if not labels_present else _yn(True),
        'notes'            : 'Training labels may be held by challenge organizers' if not labels_present else 'Check column names match validation',
        'action_needed'    : 'Request training labels from professor / organizers' if not labels_present else 'Verify column names',
    },
]

df_cmp = pd.DataFrame(rows)
print('\n=== TRAINING vs VALIDATION — COMPONENT COMPARISON ===\n')
display(df_cmp)

# Save
out_dir = Path('/content/outputs/tables')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'training_vs_validation_comparison.csv'
df_cmp.to_csv(out_path, index=False)
print(f'\nSaved to {out_path}')


=== TRAINING vs VALIDATION — COMPONENT COMPARISON ===



,component,validation_format,training_format,same,notes,action_needed
0,Run folders,Run_XXXX/ (41 runs),Run_XXXX/ (799 runs detected),Yes,Naming pattern consistent,None
1,GT csv,"Run_XXXX_GT.csv — Time + CLASS_field cols, headerle...",Run_XXXX_GT.csv — see schema cell 4a,Yes,Wide-format confirmed present,None — reuse GT parser
2,Camera timing (Visual),VisualCameraX_Run_XXXX_frame-timing.csv [Image_num...,Present,Yes,8 cameras per run expected,None
3,Camera timing (Thermal),ThermalCameraX_Run_XXXX_frame-timing.csv [Image_nu...,Present,Yes,5 thermal cameras per run expected,None
4,ISC all-timing csv,"Headerless, 7 cols: sensor_type … duration",Present — see schema cell 4c,Yes,Headerless parse required,None — reuse header=None parser
5,V2XHubSensor csv,"Headerless, 4 raw cols (unquoted [2,8] splits col 2)",Present — see schema cell 4b,Yes,4-col merge-fix required; only in runs with V2X har...,None — reuse merge-fix
6,Radar sensor JSON,List of records with topic/payload/qos/receivedAt/r...,Present — see schema cell 4d,Yes,ISO 8601 UTC timestamps; filter to traffic-objects ...,None — reuse radar parser
7,Traffic-triggers JSON,List of records with trigger_outputs → traffic_trig...,Present — see schema cell 4e,Yes,~2500 records per run; flatten to zone/lane/sensor ...,None — reuse triggers parser
8,Labels csv,conflict_no_conflict_labels_all_validation_runs.csv...,NOT FOUND (expected — withheld),Partial,Training labels may be held by challenge organizers,Request training labels from professor / organizers



Saved to /content/outputs/tables/training_vs_validation_comparison.csv


---
## 6 · Research Readiness Conclusion

In [23]:
# ── Compute high-level verdict ────────────────────────────────────────────
n_components   = len(rows)
n_same         = sum(1 for r in rows if r['same'] == 'Yes')
n_partial      = sum(1 for r in rows if r['same'] == 'Partial')
n_differ       = n_components - n_same - n_partial

all_key_files  = gt_present and isc_present and radar_present and triggers_present
run_count_ok   = len(train_run_folders_sorted) > len(val_runs)   # training should be larger

SEPARATOR = '=' * 70

print(SEPARATOR)
print(' RESEARCH READINESS VERDICT')
print(SEPARATOR)
print()
print(f'  Components matched       : {n_same} / {n_components}')
print(f'  Partial matches          : {n_partial}')
print(f'  Mismatches / missing     : {n_differ}')
print(f'  Training run count       : {len(train_run_folders_sorted)}')
print(f'  Validation run count     : {len(val_runs)}')
print(f'  Training is larger       : {run_count_ok}')
print()
print(SEPARATOR)
print(' Q1: Can the same parsing pipeline be reused?')
print(SEPARATOR)
if all_key_files and n_same >= 6:
    print('  ✓ YES — All key file types are present with matching schemas.')
    print('    The parsers from notebooks 01 & 02 can be copied directly.')
elif all_key_files:
    print('  ◑ MOSTLY — Core files present, but review flagged differences above.')
else:
    print('  ✗ PARTIAL — Some key files are missing. Manual investigation needed.')
print()
print(SEPARATOR)
print(' Q2: What changes are needed?')
print(SEPARATOR)
changes = [r['action_needed'] for r in rows if r['action_needed'] not in ('None', 'None — reuse GT parser',
    'None — reuse header=None parser', 'None — reuse merge-fix',
    'None — reuse radar parser', 'None — reuse triggers parser')]
if changes:
    for c in changes:
        print(f'  • {c}')
else:
    print('  • No structural changes needed — pipeline transfers cleanly.')
print()
print(SEPARATOR)
print(' Q3: Is training richer than validation?')
print(SEPARATOR)
if run_count_ok:
    print(f'  ✓ Yes — Training has {len(train_run_folders_sorted)} runs vs {len(val_runs)} validation runs.')
    print(f'    Richer training data available for model development.')
else:
    print(f'  ? Training has {len(train_run_folders_sorted)} runs vs {len(val_runs)} validation runs.')
    print(f'    Verify that all zip files are accessible and counted.')
print()
print(SEPARATOR)
print(' Q4: Recommended next step')
print(SEPARATOR)
if n_differ == 0 and all_key_files:
    print('  ➜ USE BOTH datasets — schemas are compatible.')
    print('    Proceed to extract training data (lightweight: skip .pcap/.mp4),')
    print('    then run the same EDA notebooks over training runs.')
    print('    Priority: GT EDA (nb03), timestamp alignment (nb04), radar (nb05).')
elif n_differ <= 2:
    print('  ➜ BUILD PARSER GENERALISATION FIRST — minor differences detected.')
    print('    Fix the flagged file types, then apply to both datasets.')
else:
    print('  ➜ CONTINUE WITH VALIDATION ONLY until training schema is confirmed.')
    print('    Request clarification / updated files from professor.')
print()
print(SEPARATOR)

 RESEARCH READINESS VERDICT

  Components matched       : 8 / 9
  Partial matches          : 1
  Mismatches / missing     : 0
  Training run count       : 799
  Validation run count     : 41
  Training is larger       : True

 Q1: Can the same parsing pipeline be reused?
  ✓ YES — All key file types are present with matching schemas.
    The parsers from notebooks 01 & 02 can be copied directly.

 Q2: What changes are needed?
  • Request training labels from professor / organizers

 Q3: Is training richer than validation?
  ✓ Yes — Training has 799 runs vs 41 validation runs.
    Richer training data available for model development.

 Q4: Recommended next step
  ➜ USE BOTH datasets — schemas are compatible.
    Proceed to extract training data (lightweight: skip .pcap/.mp4),
    then run the same EDA notebooks over training runs.
    Priority: GT EDA (nb03), timestamp alignment (nb04), radar (nb05).



---
## Appendix · Quick Reference

### Known Validation Parsing Rules (from notebook 02)

| File | Parse note |
|---|---|
| `Run_XXXX_GT.csv` | Has header (`Time`, then `CLASS_field` cols) — parse normally |
| `VisualCamera*/ThermalCamera* timing` | Has header (`Image_number`, `Timestamp`) |
| `ISC_Run_XXXX_ISC_all_timing.csv` | **Headerless** — use `header=None`, 7 cols |
| `ISC_Run_XXXX_v2xhub_timing.csv` | **Headerless** — use `header=None`, 7 cols |
| `V2XHubSensor_Run_XXXX.csv` | **Headerless, 4 raw cols** — merge cols 1..n-2 into `active_signal_ids` |
| Radar `sensorN.json` | JSON list — filter `topic` containing `traffic-objects` |
| `traffic-triggers-output.json` | JSON list — flatten `trigger_outputs → traffic_triggers` |

### Timestamp Formats

| Source | Format | TZ |
|---|---|---|
| GT `Time` | Unix float (seconds) | UTC |
| Camera `Timestamp` | `YYYY-MM-DD-HH-MM-SS_microseconds` | Local (UTC-5) |
| ISC timing `start_time` | `YYYY-MM-DD HH:MM:SS.ffffff` | Local (UTC-5) |
| V2X `timestamp_ms` | Unix integer (ms) | UTC |
| Radar / Triggers `payload.timestamp` | ISO 8601 `+00:00` | UTC |